In this unit, we describe the procedure for training the stepping detector. Further, we present the classification result for stepping detector.

# Library imports 

In [1]:
'''Import and initialize MongoClient'''
import subprocess
import os
import sys
import json
from pymongo import     MongoClient

con = MongoClient()

import pprint
import numpy as np
from pathlib import Path

In [2]:
'''Import project specific library'''
sys.path.append(os.path.join(os.getcwd(), 'LibCode'))
import HAR_OrientetionIndependence
import HARSaveToMongo
import ComputeFeaturesTransportMode
import FeaturesExtraction

In [3]:
'''For updating the lib changes effects'''
import importlib
importlib.reload(HAR_OrientetionIndependence)
importlib.reload(HARSaveToMongo)
importlib.reload(ComputeFeaturesTransportMode)
importlib.reload(FeaturesExtraction)

<module 'FeaturesExtraction' from '/home/nfsu/JRF/SoftwareExecution/Advanced-Public-Transportation-System-Software-Implementation-main/code/LibCode/FeaturesExtraction.py'>

# Save in MongoDB

The stepping detector is trained and validated on the Human Activity Recognition (HAR) data-set record [50]. The accelerometer record is stored in the `RouteName = HAR_DATASET_Preprocessing_And_HV` database of MongoDB. 

# Decomposition of accelerometer data
The accelerometer record data is decomposed to `horizontal and vertical components` with the `IntervalLength` of `140 points.`

In [4]:
def HAR_OrientetionIndependenceFunction(RouteName, IntervalLength):
    
    '''
    input: The route name and dataset directory name
    
    output: None
    
    function: It fetches the data records placed in the specified data set directory 
    and save them in the MongoDB database. The function also computes the vertical and
    horizontal accelerometer components and saves them in the MongoDB database. 
    '''        
    
    SingleTripsInfo = [LR['SingleTripInfo'] for LR in 
                       con[RouteName]['TripsInfo'].find({'ConvertedToEarthAxis':False})]

    SingleTripsInfo = HAR_OrientetionIndependence.GetSingleTripRecord(SingleTripsInfo)

    for (index,SingleTripInfo) in enumerate(SingleTripsInfo):
        AcclMagRecordsListStoppages =[AclR for AclR in 
                                      con[RouteName][f'acc.{SingleTripInfo}.Raw'].find().sort([('index',1)])]
        '''
        print(AcclMagRecordsListStoppages)
        input()
        '''

        HAR_OrientetionIndependence.ProcessEarthaxisHVComponentUsingJigSawMethod(
            RouteName,SingleTripInfo, AcclMagRecordsListStoppages, IntervalLength)

        print('Converting the accelerometer data to orientation independent record for SingleTripInfo')
        print(SingleTripInfo)
        con[RouteName]['TripsInfo'].update_one({'SingleTripInfo':f'acc.{SingleTripInfo}'},
                                               {'$set':{'ConvertedToEarthAxis':True}})

    #input()
    #'''

# Features Extraction
The feature `set-1 to set-4` are computed using the orientation independent horizontal and vertical components of accelerometer records for the window of `128 samples and 50% overlap`.

In [5]:
def ComputeFeatures(RouteName, WindowList, WindowIndex, RecordType):
    
    '''
    input: The WindowList and WindowIndex specify the window size for feature computation,
    record type (Raw or Segment other than stoppage segment), and route name.
    
    output: None
    
    function: It extracts the appropriate accelerometer components from the MongoDB database based
    on the provided input and computes the features on the windowed accelerometer component. The 
    computed features are stored in the MongoDB database.
    '''
    
    SingleTripsInfo = [LR['SingleTripInfo'] for LR in 
                       con[RouteName]['TripsInfo'].find({'ConvertedToEarthAxis':True})]

    SingleTripsInfo = HAR_OrientetionIndependence.GetSingleTripRecord(SingleTripsInfo)

    for (index,SingleTripInfo) in enumerate(SingleTripsInfo):
        HVComponent =[AclR for AclR in 
                      con[RouteName][SingleTripInfo+'.EAccHVComponent'].find(
                      ).sort([('index',1)])]
        ModeIntList = SingleTripInfo.split('.')
        ModeInt = ActivityIncluded.index(ModeIntList[0])
        #print(ModeIntList)
        #input()
        #'''
        ComputeFeaturesTransportMode.ComputeFeature(SingleTripInfo,HVComponent,ModeInt,WindowList[WindowIndex],
                                                    RecordType, RouteName)

        FeaturesExtraction.ComputeFeatureForComponents(RouteName,HVComponent,ModeInt,
                                                       WindowList[WindowIndex],SingleTripInfo,RecordType)
        #'''
        #print(SingleTripInfo)
        #print(ModeInt)
        print(SingleTripInfo)
        #pprint.pprint(AcclMagRecordsListStoppages)
        #input()


In [6]:
def LoadInMongoFromNp(RouteName, NpPathDir):
    
    '''
    input: The route name and numpy directory path
    output: The MongoDB database collections from the Numpy files
    function: It creates the MongoDB database of TripsInfo collections 
    and features collections from the Numpy files.
    '''    
    
    CollectionName = 'TripsInfo'
    TripsInfoRecords = np.load(f'{NpPathDir}/{RouteName}/{CollectionName}.npy', allow_pickle=True)
    
    print('Saving data in mongoDB')
    print(RouteName, CollectionName)
    con[RouteName][CollectionName].insert_many(TripsInfoRecords.tolist())
    
    CollectionNames = os.listdir(f'{NpPathDir}/{RouteName}')
    #print(CollectionNames)
    
    CollectionNames_1 = [rec for rec in CollectionNames if '\'.' not in rec] # To address the error
    CollectionNames_2 = [rec for rec in CollectionNames if 'Feature' in rec]
    
    #print('Saving data in mongoDB')
    for Collection in CollectionNames_2:
        RecordsList = np.load(f'{NpPathDir}/{RouteName}/{Collection}', allow_pickle=True)
        #print(Collection)
        #pprint.pprint(RecordsList[0:3])
        
        CollectionName = Collection[0:-4]
        
        print(RouteName, CollectionName)
        con[RouteName][CollectionName].insert_many(RecordsList.tolist())
        
    
    
def SaveInNp(RouteName, NpPathDir):
    
    '''
    input: The route name and numpy directory path
    output: Numpy files of the MongoDB database
    function: It stores the Numpy files for the MongoDB database in the specified directory path
    '''    
    
    CollectionNames = [Collection for Collection in 
                        con[RouteName].list_collection_names() if Collection!='system.indexes']

    for CollectionName in CollectionNames:
        print('CollectionName', CollectionName)
        RecordsList = [rec for rec in con[RouteName][CollectionName].find().sort([('_id',1)])]

        for RecordDict in RecordsList:
            del[RecordDict['_id']]

        if os.path.exists(os.path.join(NpPathDir, RouteName)) == False:
            os.mkdir(os.path.join(NpPathDir, RouteName))

            
        #np.save(f'{Path}/{Database}/{CollectionName}.npy', RecordsList)
        np.save(os.path.join(NpPathDir, RouteName,f'{CollectionName}.npy'), RecordsList)
        

In [7]:
'''For directory management'''
path = Path(os.getcwd())
OneLevelUpPath = path.parents[0]

In [8]:
'''Path for Np'''
NpPathDir = os.path.join(str(OneLevelUpPath), 'data','NpData')

## Variables

`ProjectDataUsed`: determines whether the project data or the user's own data is used for execution.

`UsedPreTrained`: determines whether the pretrained and precomputed dataset or raw data is used for execution.

`UseMongoDB`: determines whether the MonngoDB database or Numpy file is used for execution.

`ReducedKFolds`: If false: one fold is used, else ten-fold is used

In [9]:
'''
ProjectDataUsed = True
UsedPreTrained = True
ReducedKFolds = False
UseMongoDB = True
'''

#'''
ProjectDataUsed = True
UsedPreTrained = True
ReducedKFolds = True
UseMongoDB = False
#'''

'''
ProjectDataUsed = True
UsedPreTrained = False
ReducedKFolds = False
UseMongoDB = False
'''

'\nProjectDataUsed = True\nUsedPreTrained = False\nReducedKFolds = False\nUseMongoDB = False\n'

In [10]:
'''Variable initialization'''
RouteName = 'Demo_HAR_DATASET_Preprocessing_And_HV'
if ProjectDataUsed==True:
    RecordDir = os.path.join(str(OneLevelUpPath), 'data','HAPTDataSet','RawData','')
else:
    RecordDir = os.path.join(str(OneLevelUpPath), 'data','UserData','HAPTDataSet','RawData','')
IntervalLength = 140
RecordType = '.Raw'

In [11]:
WindowList = [32,64,128,256,512]
WindowIndex = 2
Activties = ['WALKING', 'WALKING_UPSTAIRS','WALKING_DOWNSTAIRS','SITTING','STANDING','LAYING',
            'STAND_TO_SIT','SIT_TO_STAND','SIT_TO_LIE','LIE_TO_SIT','STAND_TO_LIE','LIE_TO_STAND']
ActivityID = [1,2,3,4,5,6,7,8,9,10,11,12]
ActivityIncluded = ['WALKING', 'WALKING_UPSTAIRS','WALKING_DOWNSTAIRS','SITTING','STANDING','LAYING',
            'STAND_TO_SIT','SIT_TO_STAND','SIT_TO_LIE','LIE_TO_SIT','STAND_TO_LIE','LIE_TO_STAND']
ActivityIncludedClassInt = [0,1,2,3,4]

In [12]:
if UsedPreTrained==False and UseMongoDB==True:
    
    print('Reading data and saving in MongoDB')
    HARSaveToMongo.SaveHARDataInMongo(RouteName, RecordDir)
    print(f'Computing preprocessing for {RecordType} segments')
    HAR_OrientetionIndependenceFunction(RouteName, IntervalLength)
    print(f'Computing features for {RecordType} segments')
    ComputeFeatures(RouteName, WindowList, WindowIndex, RecordType)
    
    
    print('Saving MongoData in Np files')
    SaveInNp(RouteName, NpPathDir)


elif UseMongoDB==True:
    RouteNamesListInDB = con.list_database_names()
    if RouteName not in RouteNamesListInDB:
        '''Load the data for RouteName, if RouteName is not in RouteNamesList'''
        print('Loading MongoData from Np files')
        LoadInMongoFromNp(RouteName, NpPathDir)
    

In [13]:
#con.drop_database(RouteName)

# Classification 

In [14]:
'''Import Libs'''
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix,precision_score, recall_score, f1_score
from sklearn import tree
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from collections import Counter
from sklearn.impute import SimpleImputer

'''Custom library'''
import CAC_Helper
import Stepper_Helper

In [15]:
'''For updating the lib changes effects'''
import importlib
importlib.reload(CAC_Helper)
importlib.reload(Stepper_Helper)

<module 'Stepper_Helper' from '/home/nfsu/JRF/SoftwareExecution/Advanced-Public-Transportation-System-Software-Implementation-main/code/LibCode/Stepper_Helper.py'>

In [16]:
'''Variables for Classification'''
ResultPathDir = os.path.join(str(OneLevelUpPath), 'results','SteppingDetector','')


if os.path.exists(ResultPathDir) == False:
    os.mkdir(ResultPathDir)

TrainedModelPathDir = os.path.join(str(OneLevelUpPath), 'data', 'TrainedModel','SteppingDetector','')

'''Any one route type as decided by Stoppage experiment'''
#RecordType = '.SegmentOtherThanStoppage'
#RecordType = '.StoppageSegment'
RecordType = '.Raw'

ClassifierList = [GaussianNB(), LogisticRegression(random_state=0), 
                  RandomForestClassifier(max_depth=20), tree.DecisionTreeClassifier(), 
                  SVC(gamma='auto')
                 ]
ClassifierNameList = ['NB', 'LogisticRegression', 'RF', 'DT', 'SVC']

Index = 3

Classifier = ClassifierList[Index]
ClassifierName = ClassifierNameList[Index]

In [17]:
def ApplyClassification(Classes, FeatureType, RecordType, SelectedFeatures, SelectedFeaturesFlag,
                        Classifier, ClassifierName, ResultPathDir, ProjectDataUsed, RouteName, 
                        UsedPreTrained, ClassifierName_ForModelSave, TrainedModelPathDir, ReducedKFolds,
                        NpPathDir, UseMongoDB
                       ):
    
    '''
    input: 
        Classes: The number of classes
        
        FeatureType and RecordType: feature type and record type information
        
        SelectedFeatures and SelectedFeaturesFlag: selected features list and flag to 
        determine if selected features are list is used
        
        Classifier and ClassifierName: classifier object variable and classifier name
        
        ResultPathDir: path of result directory
        
        ProjectDataUsed: flag to determine whether the project dataset or user dataset is used
        
        RouteNameTestList: The route names list to be considered during classification
        
        UsedPreTrained: Flag to determine whether the pretrained classifier should be used
        or the classifier is to be trained
        
        ClassifierName_ForModelSave: the name of classifier for saving a model
        
        TrainedModelPathDir: path of trainedModel directory 
    
    output: None
    
    function: It trains and validates the classifier the ten-fold cross-validation with stratified
    sampling of each class. The performance metrics of the classifier is stores as a txt file in the
    result directory
    
    '''        
    
    MetricsDict = CAC_Helper.InitializeMetricsDict(Classes)
    
    #SelectedFeatures = []
    
    if UseMongoDB==True:
        #X, y = Stepper_Helper.GetFeaturesFromMongoDB(RouteName, FeatureType, RecordType)
        X, y = Stepper_Helper.GetFeaturesFromMongoDB(RouteName, FeatureType, RecordType, 
                                                     SelectedFeatures, SelectedFeaturesFlag)
        
        
        if os.path.exists(os.path.join(NpPathDir,'SteppingDetector'))==False:
            os.mkdir(os.path.join(NpPathDir,'SteppingDetector'))
            
        np.save(f'{NpPathDir}/SteppingDetector/X_{FeatureType}_{RecordType}_{SelectedFeaturesFlag}.npy', X)
        np.save(f'{NpPathDir}/SteppingDetector/y_{FeatureType}_{RecordType}_{SelectedFeaturesFlag}.npy', y)

    else:
        X = np.load(f'{NpPathDir}/SteppingDetector/X_{FeatureType}_{RecordType}_{SelectedFeaturesFlag}.npy',
                    allow_pickle=True)
        y = np.load(f'{NpPathDir}/SteppingDetector/y_{FeatureType}_{RecordType}_{SelectedFeaturesFlag}.npy',
                    allow_pickle=True)
    
    
    MetricsDict = CAC_Helper.TrainAndPredict(X, y, Classifier, MetricsDict, ResultPathDir,
                                             ClassifierName, UsedPreTrained, 
                                             ClassifierName_ForModelSave, TrainedModelPathDir, ReducedKFolds)
    
    CAC_Helper.PrintMetricsDict(ClassifierName, ResultPathDir, FeatureType, RecordType, 
                                SelectedFeaturesFlag, MetricsDict)

In [18]:
Classes = 5


In [19]:
for Classifier, ClassifierName in zip(ClassifierList, ClassifierNameList):
    print('Classifier, ClassifierName', Classifier, ClassifierName)
    #for RecordType in ['.Raw', '.SegmentOtherThanStoppage']:

    '''Feature set-1'''
    FeatureType = '.HARFeature'
    SelectedFeaturesFlag = False
    SelectedFeatures = []

    print('Feature set-1')
    ClassifierName_ForModelSave = ClassifierName+"_Set1"
    ApplyClassification(Classes, FeatureType, RecordType, SelectedFeatures, SelectedFeaturesFlag,
                        Classifier, ClassifierName, ResultPathDir, ProjectDataUsed, RouteName,
                        UsedPreTrained, ClassifierName_ForModelSave, TrainedModelPathDir, ReducedKFolds, 
                        NpPathDir, UseMongoDB
                       )

    '''Feature set-2'''
    FeatureType = '.HARFeature'
    SelectedFeaturesFlag = True
    ClassifierName_ForModelSave = ClassifierName+"_Set2"
    SelectedFeatures = CAC_Helper.SelectedFeaturesForFeatureType(FeatureType)

    print('Feature set-2')

    ApplyClassification(Classes, FeatureType, RecordType, SelectedFeatures, SelectedFeaturesFlag,
                        Classifier, ClassifierName, ResultPathDir, ProjectDataUsed, RouteName, 
                        UsedPreTrained, ClassifierName_ForModelSave, TrainedModelPathDir, ReducedKFolds, 
                        NpPathDir, UseMongoDB
                       )

    '''Feature set-3'''
    FeatureType = '.TransportFeatures'
    SelectedFeatures = []
    SelectedFeaturesFlag = False
    print('Feature set-3')
    ClassifierName_ForModelSave = ClassifierName+"_Set3"

    ApplyClassification(Classes, FeatureType, RecordType, SelectedFeatures, SelectedFeaturesFlag,
                        Classifier, ClassifierName, ResultPathDir, ProjectDataUsed, RouteName,
                        UsedPreTrained, ClassifierName_ForModelSave, TrainedModelPathDir, ReducedKFolds, 
                        NpPathDir, UseMongoDB
                       )   


    '''Feature set-4'''
    FeatureType = '.TransportFeatures'
    SelectedFeatures = CAC_Helper.SelectedFeaturesForFeatureType(FeatureType)
    SelectedFeaturesFlag = True

    print('Feature set-4')
    ClassifierName_ForModelSave = ClassifierName+"_Set4"
    ApplyClassification(Classes, FeatureType, RecordType, SelectedFeatures, SelectedFeaturesFlag,
                        Classifier, ClassifierName, ResultPathDir, ProjectDataUsed, RouteName,
                        UsedPreTrained, ClassifierName_ForModelSave, TrainedModelPathDir, ReducedKFolds, 
                        NpPathDir, UseMongoDB
                       )        



Classifier, ClassifierName GaussianNB() NB
Feature set-1
Feature set-2
Feature set-3
Feature set-4
Classifier, ClassifierName LogisticRegression(random_state=0) LogisticRegression
Feature set-1
Feature set-2
Feature set-3


/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behav

Feature set-4
Classifier, ClassifierName RandomForestClassifier(max_depth=20) RF
Feature set-1
Feature set-2
Feature set-3
Feature set-4
Classifier, ClassifierName DecisionTreeClassifier() DT
Feature set-1


/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behav

Feature set-2
Feature set-3
Feature set-4
Classifier, ClassifierName SVC(gamma='auto') SVC
Feature set-1


/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behav

Feature set-2
Feature set-3
Feature set-4


/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nfsu/JRF/SoftwareExecution/Softwar_Env/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behav

In [20]:
'''Read the value for one of the machine learning algorithm'''
file = os.path.join(ResultPathDir,f'{ClassifierName}.txt')
f = open(file, "r")
print(f.read())

Results for .Raw, .HARFeature, and selected features flag: False
ConfusionMatrix 
[[4.668e+03 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [3.000e+00 0.000e+00 0.000e+00 1.523e+03 2.710e+02]
 [2.000e+00 0.000e+00 0.000e+00 2.620e+02 1.713e+03]]
 
PrecissionValue 
[0.99893162 0.         0.         0.85604504 0.86388957]
 
RecallValue 
[1.         0.         0.         0.84747052 0.86640004]
 
F1ScoreValue 
[0.99946524 0.         0.         0.85073667 0.8643697 ]
 
AccuracyValue 
0.9362667208839281
 
Results for .Raw, .HARFeature, and selected features flag: True
ConfusionMatrix 
[[4.668e+03 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [4.000e+00 0.000e+00 0.000e+00 1.507e+03 2.860e+02]
 [3.000e+00 0.000e+00 0.000e+00 2.620e+02 1.712e+03]]
 
PrecissionValue 
[0.99850791 0.         0.  